# Chapter 12 — Capstone: End-to-End ML Workflow

**Session 4 | Final Chapter | Exercise time: ~35 minutes**

---

## The Task

April 15, 1912. The RMS Titanic sinks. 1,502 of 2,224 passengers die.

**Can we predict who survived, based on passenger characteristics?**

You will run a **complete, professional ML workflow** from raw data to interpreted results:

```
① Load & Explore  →  ② Clean & Preprocess  →  ③ Feature Engineering
④ Train Models    →  ⑤ Evaluate & Compare  →  ⑥ Interpret & Reflect
```

**Primary metric:** F1-score (balances precision and recall)

> Work through the steps in order. Each builds on the previous one.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
np.random.seed(42)

print('All imports ready!')

---
## Step 1 — Load and Explore the Data
**⏱ ~5 minutes**

The Titanic dataset is built into seaborn — no download needed.

In [ ]:
# Load the dataset
df_raw = sns.load_dataset('titanic')
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
# TODO 1a: Check the data types and missing values
# Hint: df.dtypes and df.isnull().sum()

# Your code here:


In [ ]:
# TODO 1b: Visualize survival rate by gender
# Hint: use sns.barplot with x='sex', y='survived'
# What do you notice?

# Your code here:


In [ ]:
# TODO 1c: Visualize survival rate by passenger class (pclass)
# Hint: use sns.barplot with x='pclass', y='survived'

# Your code here:


In [ ]:
# TODO 1d: Plot the age distribution of survivors vs non-survivors
# Hint: use two sns.histplot calls (one per survived group), or
#       sns.histplot with hue='survived'

# Your code here:


**Quick check:** Before moving on, answer these:
- Which gender had a higher survival rate?
- Which class survived the most? The least?
- Which columns have significant missing values?

---
## Step 2 — Clean and Preprocess
**⏱ ~8 minutes**

We'll work on a copy so the raw data stays intact.

In [ ]:
# Start from a clean copy
df = df_raw.copy()

# The seaborn titanic dataset has both 'sex'/'embarked' (string)
# and 'pclass'/'survived' (int). Let's check what we have:
print(df.columns.tolist())
print(df[['sex', 'embarked', 'pclass', 'survived']].dtypes)

In [ ]:
# TODO 2a: Impute missing Age values
#
# Strategy: fill each passenger's missing age with the MEDIAN age
# of passengers in the SAME pclass.
# (This is better than a single global median)
#
# Hint: use df.groupby('pclass')['age'].transform('median')
#       then use fillna() to fill missing values

# Your code here:


# Check: how many missing age values remain?
print('Missing age values after imputation:', df['age'].isnull().sum())

In [ ]:
# TODO 2b: Fill missing Embarked values
# Strategy: fill with the mode (most common port)
# Hint: df['embarked'].mode()[0]

# Your code here:


print('Missing embarked values:', df['embarked'].isnull().sum())

In [ ]:
# TODO 2c: Drop columns we won't use
#
# Drop: 'deck' (like Cabin — too sparse)
#       'embark_town' (duplicate of 'embarked')
#       'alive' (string version of 'survived' — data leakage!)
#       'who', 'adult_male', 'alone' (derived columns — use raw features)
#
# Hint: df.drop(columns=[...], inplace=True)

# Your code here:


print('Remaining columns:', df.columns.tolist())

In [ ]:
# TODO 2d: Encode categorical variables
#
# 1. 'sex' → binary: 'male' = 1, 'female' = 0
#    Hint: df['sex'] = (df['sex'] == 'male').astype(int)
#
# 2. 'embarked' → one-hot encode (3 values: C, Q, S)
#    Hint: pd.get_dummies(df, columns=['embarked'], drop_first=True)
#    (drop_first=True drops one dummy to avoid multicollinearity)

# Your code here:


print('Shape after encoding:', df.shape)
df.head(3)

---
## Step 3 — Feature Engineering
**⏱ ~5 minutes**

Create new features that capture meaningful patterns.

In [ ]:
# TODO 3a: Create FamilySize feature
# FamilySize = number of siblings/spouses + parents/children + the passenger themselves
# Formula: SibSp + Parch + 1

# Your code here:


print('FamilySize distribution:')
print(df['family_size'].value_counts().sort_index())

In [ ]:
# TODO 3b: Create IsAlone feature
# IsAlone = 1 if the passenger was traveling alone (FamilySize == 1), else 0

# Your code here:


# Quick check: what fraction of passengers were alone?
print(f"Fraction traveling alone: {df['is_alone'].mean():.1%}")

In [ ]:
# TODO 3c: Visualize survival rate by FamilySize
# Does traveling with family help or hurt survival odds?

# Your code here:


---
## Step 4 — Train and Evaluate Multiple Models
**⏱ ~12 minutes**

Now the fun part: train three classifiers and compare them.

In [ ]:
# TODO 4a: Prepare X and y
# X = all columns except 'survived'
# y = 'survived'
#
# Then wrap scaling into a Pipeline so there is no leakage during cross-validation:
#   pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
# You will use this pattern in TODO 4b.

# Your code here:


print('X shape:', X.shape)
print('y shape:', y.shape)
print('Survival rate:', y.mean().round(3))

In [ ]:
# TODO 4b: Cross-validate three models using Pipelines (StandardScaler + classifier)
#
# Use 5-fold cross-validation with cv=5
# Evaluate with scoring='f1' AND scoring='accuracy'
#
# Models to compare:
#   - LogisticRegression(max_iter=1000, random_state=42)
#   - RandomForestClassifier(n_estimators=100, random_state=42)
#   - GradientBoostingClassifier(n_estimators=100, random_state=42)
#
# Wrap each in a Pipeline([('scaler', StandardScaler()), ('clf', model)])
# Store results in a dictionary or a DataFrame for easy comparison

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Your cross-validation code here:


# Print a comparison table


In [ ]:
# TODO 4c: Plot the comparison as a bar chart
# Show both F1 and Accuracy side by side for each model
# Include error bars (std from cross-validation)

# Your code here:


---
## Step 5 — Interpret the Results
**⏱ ~5 minutes**

In [ ]:
# TODO 5a: Feature importances from your best model
#
# Use the model with the highest F1-score from Step 4b
# (typically Random Forest or Gradient Boosting).
#
# Random Forest does NOT need scaling for feature importances — fit directly on X:
#   rf = RandomForestClassifier(n_estimators=100, random_state=42)
#   rf.fit(X, y)
#
# Then plot feature_importances_ as a sorted horizontal bar chart.

# Your code here:


In [ ]:
# TODO 5b: Plot a confusion matrix for the best model
#
# 1. Split into train/test (test_size=0.2, random_state=42)
# 2. Fit your best model on the training set
# 3. Predict on the test set
# 4. Plot the confusion matrix with ConfusionMatrixDisplay
#
# Then answer: how many False Negatives are there?
# (predicted 'did not survive', actually survived)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Your code here:


In [ ]:
# TODO 5c: Print the full classification report
# Hint: classification_report(y_test, y_pred)
# What are the precision and recall for each class?

# Your code here:


---
## Step 6 — Reflect

Take 2 minutes and discuss (or write down your thoughts):

1. **Which feature had the highest importance?** Does that match what you saw in the EDA?

2. **What does a False Negative mean here?** (Predicted 'died', actually survived.) How serious is that error?

3. **Which preprocessing step was most important?** Imputing age? Encoding sex? Engineering FamilySize?

4. **If you had more time, what would you try next?**
   - Hyperparameter tuning?
   - Extract titles from names (Mr., Mrs., Master, Miss)?
   - Try a different model?

In [ ]:
# Write your 3–4 sentence reflection here.
# What were the 2–3 most important findings from your analysis?
# What surprised you? What would you try next?

reflection = """
Key findings:
1. ...
2. ...
3. ...

What surprised me: ...

What I would try next: ...
"""
print(reflection)

---

## Well done.

You just ran a complete, professional ML workflow — the same steps that data scientists use in the real world.

**This is what Chapter 1 called the Data Science workflow. Now you've lived it.**

---

**Solutions:** See `../04-solutions/ch12_capstone_solutions.ipynb`